Problem: Implement Binary Focal Loss
Given n binary classification samples, true labels y_i ∈ {0,1}, and predicted probabilities p_i for the positive class, implement Binary Focal Loss.

For each sample:

If y_i = 1, then p_t = p_i, alpha_t = alpha.
If y_i = 0, then p_t = 1 - p_i, alpha_t = 1 - alpha.
The per-sample loss is:

loss_i = - alpha_t * (1 - p_t)^gamma * log(p_t)
To avoid log(0), clip probabilities to [1e-15, 1 - 1e-15].

Input Format
n gamma alpha reduction
y_1 y_2 ... y_n
p_1 p_2 ... p_n
Where:

reduction is one of mean, sum, or none.
mean outputs the average loss.
sum outputs the total loss.
none outputs every sample's loss.
Output Format
If reduction is mean or sum, print one float rounded to 6 decimal places.
If reduction is none, print n floats separated by spaces, each rounded to 6 decimal places.
Constraints
1 <= n <= 100000
0 <= gamma <= 10
0 <= alpha <= 1
0 <= p_i <= 1
Example
Input:

3 2.0 0.25 mean
1 0 1
0.9 0.1 0.2
Output:

0.086188
Example
Input
3 2.0 0.25 mean
1 0 1
0.9 0.1 0.2
Output
0.086188


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [44]:
import torch

CLIP_EPS = 1e-15

def binary_focal_loss(y, p, gamma, alpha, reduction="mean"):
    """
    Binary focal loss (Lin et al., 2017), probability inputs.

    Args:
        y: labels in {0, 1}, shape (n,). May be an int or float tensor / list.
        p: predicted prob of the positive class, shape (n,), values in [0, 1].
        gamma: float >= 0, focusing parameter.
        alpha: float in [0, 1], weight on the positive class.
        reduction: "mean" | "sum" | "none".

    Returns:
        torch.Tensor, dtype float64.
          reduction="mean"/"sum" -> 0-dim scalar tensor
          reduction="none"       -> shape (n,)

    TODO:
      1. Coerce inputs with torch.as_tensor(..., dtype=torch.float64).
         Use as_tensor, not torch.tensor: as_tensor is a no-copy view when
         possible and preserves the autograd graph if p already requires grad.
      2. Clamp p into [CLIP_EPS, 1 - CLIP_EPS] (torch.clamp) before any log.
      3. p_t     = p     where y == 1, else 1 - p
         alpha_t = alpha where y == 1, else 1 - alpha
         Use torch.where on a boolean mask. No Python loop: n can be 1e5.
      4. loss = -alpha_t * (1 - p_t) ** gamma * torch.log(p_t)
      5. Reduce: .mean() / .sum() / return as-is.
         Raise ValueError for anything else.

    Keep every op out-of-place so this stays differentiable w.r.t. p.
    """
    y, p = torch.tensor(y, dtype=torch.float64), torch.tensor(p, dtype=torch.float64)
    p = p.clamp(CLIP_EPS, 1 - CLIP_EPS)
    alpha = alpha * y + (1-alpha) * (1-y)
    pt = p * y + (1 - p) * (1-y)
    # loss_1 = alpha * ((1 - p) * y)**gamma * torch.log((1 - p) * y )
    # loss_0 = (1-alpha) * (p * (1-y))**gamma * torch.log(p * (1-y))
    loss = -1.0 * (alpha * (1-pt) ** gamma * torch.log(pt))
    if reduction == "mean":
      return loss.mean()
    elif reduction == "sum":
      return loss.sum()
    else:
      return loss
    # raise NotImplementedError


def solve(input_str):
    """
    Parse the stdin format and return the stdout string.

    Input (3 lines):
        n gamma alpha reduction
        y_1 ... y_n
        p_1 ... p_n

    Return:
        "mean"/"sum": one float, 6 decimals
        "none":       n floats, 6 decimals, space separated

    TODO:
      1. Split lines, parse header (n int, gamma/alpha float, reduction str).
      2. Parse the two numeric rows.
      3. Call binary_focal_loss, then pull numbers out of the tensor
         (.item() for scalars, .tolist() for the "none" case) and format
         with f"{v:.6f}".
    """
    lines = [ln for ln in input_str.strip().splitlines() if ln.strip()]
    header = lines[0].split()
    n = int(header[0])
    gamma = float(header[1])
    alpha = float(header[2])
    reduction = header[3]
    y = [int(v) for v in lines[1].split()]
    p = [float(v) for v in lines[2].split()]
    assert len(y) == n and len(p) == n, f"expected {n} values, got {len(y)} lables and {len(p)} probs"
    out = binary_focal_loss(y, p, gamma, alpha, reduction)
    if reduction == "none":
      return " ".join(f"{v:.6f}" for v in out.tolist())
    return f"{out.item():.6f}"
    # raise NotImplementedError

In [45]:

"""
Test harness for the Binary Focal Loss stdin/stdout problem.

Usage in Colab:
    1. Put your solution() in a cell (it should read sys.stdin and print).
    2. Paste this cell below it.
    3. Call run_tests(solution)

Notes:
  - Only the LAST non-empty line of your output is checked, so leftover
    debug prints are tolerated.
  - Values are compared numerically with tol=5e-7, not by string equality,
    so a trailing "-0.000000" or "0.0" style difference will not fail you.
"""

import io
import sys
import time

# (stdin_text, expected_values)
TESTS = [
    # name, input, expected
    ("example / mean",
     "3 2.0 0.25 mean\n1 0 1\n0.9 0.1 0.2\n",
     [0.086188]),

    ("example / sum",
     "3 2.0 0.25 sum\n1 0 1\n0.9 0.1 0.2\n",
     [0.258564]),

    ("example / none",
     "3 2.0 0.25 none\n1 0 1\n0.9 0.1 0.2\n",
     [0.000263, 0.000790, 0.257510]),

    ("n=1, gamma=0 (plain BCE, alpha=0.5)",
     "1 0.0 0.5 mean\n1\n0.5\n",
     [0.346574]),

    ("gamma=0 collapses to weighted BCE, symmetric alpha",
     "4 0.0 0.5 none\n1 0 1 0\n0.9 0.9 0.1 0.1\n",
     [0.052680, 1.151293, 1.151293, 0.052680]),

    ("alpha=1 zeroes out negatives",
     "2 2.0 1.0 none\n1 0\n0.7 0.3\n",
     [0.032101, 0.000000]),

    ("alpha=0 zeroes out positives",
     "2 2.0 0.0 none\n1 0\n0.7 0.3\n",
     [0.000000, 0.032101]),

    ("clipping: p=0 with y=1, p=1 with y=1, p=0 with y=0",
     "3 2.0 0.25 none\n1 1 0\n0.0 1.0 0.0\n",
     [8.634694, 0.000000, 0.000000]),

    ("gamma=10, mixed confidences",
     "5 10.0 0.75 none\n1 1 0 0 1\n0.5 0.01 0.99 0.5 0.999\n",
     [0.000508, 3.123625, 1.041208, 0.000169, 0.000000]),

    ("single negative, gamma=5, sum",
     "1 5.0 0.3 sum\n0\n0.5\n",
     [0.015163]),

    ("non-round gamma, 6 samples, mean",
     "6 1.5 0.4 mean\n0 1 1 0 1 0\n0.5 0.5 0.75 0.25 0.33 0.66\n",
     [0.145216]),
]

TOL = 5e-7


def _capture(fn, stdin_text):
    """Run fn() with stdin_text piped in, return everything it printed."""
    old_in, old_out = sys.stdin, sys.stdout
    sys.stdin = io.StringIO(stdin_text)
    sys.stdout = buf = io.StringIO()
    try:
        fn()
    finally:
        sys.stdin, sys.stdout = old_in, old_out
    return buf.getvalue()


def _last_line(text):
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return lines[-1] if lines else ""


def run_tests(fn, verbose=True):
    passed = 0
    for name, stdin_text, expected in TESTS:
        try:
            raw = _capture(fn, stdin_text)
        except Exception as e:
            print(f"[ERROR] {name}\n        raised {type(e).__name__}: {e}")
            continue

        line = _last_line(raw)
        try:
            got = [float(tok) for tok in line.split()]
        except ValueError:
            print(f"[FAIL ] {name}\n        last line not parseable as floats: {line!r}")
            continue

        if len(got) != len(expected):
            print(f"[FAIL ] {name}\n        expected {len(expected)} value(s), got {len(got)}: {line!r}")
            continue

        bad = [i for i, (g, e) in enumerate(zip(got, expected)) if abs(g - e) > TOL]
        if bad:
            print(f"[FAIL ] {name}")
            print(f"        expected: {' '.join(f'{v:.6f}' for v in expected)}")
            print(f"        got     : {' '.join(f'{v:.6f}' for v in got)}")
            print(f"        first mismatch at index {bad[0]}")
            continue

        passed += 1
        if verbose:
            print(f"[PASS ] {name}")

    print(f"\n{passed}/{len(TESTS)} passed")
    return passed == len(TESTS)


def run_timing(fn, n=100000):
    """Not a correctness check. Just confirms 1e5 samples is not slow."""
    import random
    random.seed(0)
    y = " ".join(random.choice("01") for _ in range(n))
    p = " ".join(f"{random.random():.6f}" for _ in range(n))
    stdin_text = f"{n} 2.0 0.25 mean\n{y}\n{p}\n"

    t0 = time.perf_counter()
    raw = _capture(fn, stdin_text)
    dt = time.perf_counter() - t0
    print(f"n={n} finished in {dt:.3f}s, output: {_last_line(raw)}")
    if dt > 2.0:
        print("  ^ slower than expected; check for a Python-level loop over samples")


if __name__ == "__main__":
    print("Import this module and call run_tests(your_solution_fn).")

Import this module and call run_tests(your_solution_fn).


In [46]:
import sys

def main():
    print(solve(sys.stdin.read()))

run_tests(main)
run_timing(main)

[PASS ] example / mean
[PASS ] example / sum
[PASS ] example / none
[PASS ] n=1, gamma=0 (plain BCE, alpha=0.5)
[PASS ] gamma=0 collapses to weighted BCE, symmetric alpha
[PASS ] alpha=1 zeroes out negatives
[PASS ] alpha=0 zeroes out positives
[PASS ] clipping: p=0 with y=1, p=1 with y=1, p=0 with y=0
[PASS ] gamma=10, mixed confidences
[PASS ] single negative, gamma=5, sum
[PASS ] non-round gamma, 6 samples, mean

11/11 passed
n=100000 finished in 0.108s, output: 0.304670
